# Project 5: Using Agent Frameworks: Pydantic AI



In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp.client.transports import StdioTransport

load_dotenv(override=True)

True

## Step 1: Create the agent

In Pydantic AI an agent is an `Agent`: a model string plus `instructions`, which is its system prompt. The string `"openai-chat:gpt-5.4-mini"` picks the provider and the model, with `OPENAI_API_KEY` coming from the repo-root `.env`. The `-chat` names OpenAI's Chat Completions API explicitly. A bare `openai:` works too, but its default shifts in the coming Pydantic AI v2, and Chat Completions is the API every OpenAI-compatible endpoint speaks, which is what we want for the model swap at the end of the day.

In [2]:
MODEL = "openai-chat:gpt-5.4-mini"

agent = Agent(
    MODEL,
    instructions="You are a concise, friendly assistant. Reply in a single short sentence.",
)

## Step 2: Run it

Send a message, await the reply, and print the result's `.output`. With no tools yet there is nothing to loop over, so the agent just answers; this is still only an LLM call. There is a synchronous `agent.run_sync(...)` too, but inside a notebook we await `agent.run(...)` so it cooperates with Jupyter's event loop.

In [3]:
result = await agent.run("Say hello in Spanish.")
print(result.output)

Hola.


## Using SQLite todo board

The worker coordinates through the same tiny SQLite board as Day 1, the same `board.py` file: one file, one table, no server to run. A worker is handed one **goal**; to reach it, it writes its own **step** todos under that goal, ticks each one off as it goes, and closes the goal at the end. Under the hood the board is just a list of dicts (a goal has `parent_id` of None, a step points at its goal):

In [5]:
import board

board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.',
  'status': 'pending',
  'result': ''}]

`show_board()` prints that same data nicely, in the rich style from Week 1: each goal with its steps indented beneath, done tasks struck through in green and the one in progress in yellow. There are no steps yet; the worker writes its own when it plans.

In [6]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.

## Step 3: Add tools

A tool in Pydantic AI is a plain typed Python function. Pydantic reads the type hints and the docstring and builds the JSON schema for you, so there is nothing else to declare. The signature idiom is the `@agent.tool` decorator; passing the same functions in `tools=[...]` does exactly the same thing and lets us reuse them across agents, which is what we do here.

We write three small board tools: `show_todos` to read the board, `plan_steps` to break a goal into steps, and `complete_task` to mark a todo done. Here we give a quick agent just two of them and ask what is on the board; watch it decide, on its own, to call `show_todos` before it answers. That decide, call, read, answer is the agent loop starting to turn. All three tools come together in step 5.

In [7]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [8]:
board_agent = Agent(
    MODEL,
    instructions="You help manage a shared todo board.",
    tools=[show_todos, complete_task],
)

In [9]:
result = await board_agent.run("What is on the board right now, and what is its status?")
print(result.output)

There is 1 item on the board:

- **Task 1**: “Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.”
  - **Status**: pending


## Step 4: Add MCP

MCP is just more tools: ones you did not write, connected over a small protocol. We give the agent the filesystem reference server, the same Node server every framework uses this week, scoped to a single `workspace` folder so it can only touch files in there. In Pydantic AI an MCP server is a toolset: you wrap it in an `MCPToolset`, pass it via `toolsets=[...]`, and open its connection with `async with agent:` around the run.

We point `log_file` at the null device (`Path(os.devnull)`). That keeps the server's startup logging off the screen and, more importantly, lets the server run from a Jupyter kernel on Windows, where the kernel's own stderr has no real file descriptor for the server to write to. On Mac and Linux it simply keeps the output clean.

In [10]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch

filesystem = MCPToolset(
    StdioTransport(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
        cwd=str(workspace),  # start the server in the workspace so relative file names resolve there
        log_file=Path(os.devnull),
    ), 
    init_timeout=60
)

In [11]:
file_agent = Agent(
    MODEL,
    instructions="You can read and write files in your workspace. Use your tools to do what is asked.",
    toolsets=[filesystem],
)

In [13]:
async with file_agent:
    result = await file_agent.run("Read notes.txt and summarize it in one short sentence.")
print(result.output)

The notes welcome the team and describe building a small language tutor collaboratively, one careful task at a time.


## Step 5: Put it in a loop with a goal

Give one agent all three board tools and the filesystem server, hand it the goal, and let it run. It plans its own steps on the board, works them with its file tools, ticks each one off, and closes the goal when the work is done. That is the agent loop running on its own: read, plan, act, check off, repeat. Watch the board fill with steps and then strike them through.

In [14]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    MODEL,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task],
    toolsets=[filesystem],
)

board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.")
board.claim_todo(goal_id)

async with worker:
    result = await worker.run("Please work the pending goal on the board.")
print(result.output)
board.show_board()

Done — I translated `notes.txt` into natural Spanish and saved it as `spanish.txt`.


Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to spanish.txt.  Completed the translation task and created spanish.txt.
  Step #2: Inspect notes.txt to understand the source text.  Reviewed notes.txt and understood the source text.
  Step #3: Translate the text into natural Spanish.  Translated the notes into natural Spanish.
  Step #4: Write the Spanish translation to spanish.txt.  Wrote the Spanish translation to spanish.txt.
  Step #5: Verify the output and mark the goal complete.  Verified the translation output was written successfully.